# Taller Streaming y Kafka â€” Etapas 1, 2 y 3

**Dataset:** `online_retail_II.csv` â€” transacciones de una tienda online del Reino Unido (2009â€“2011).

| Columna | DescripciÃ³n |
|---|---|
| `Invoice` | NÃºmero de factura (si empieza por `C` es una cancelaciÃ³n) |
| `StockCode` | CÃ³digo del producto |
| `Description` | Nombre del producto |
| `Quantity` | Unidades vendidas (negativo = devoluciÃ³n) |
| `InvoiceDate` | Fecha y hora de la factura |
| `Price` | Precio unitario |
| `Customer ID` | Identificador del cliente |
| `Country` | PaÃ­s del cliente |

En este notebook se desarrollan:
1. **Etapa 1:** SesiÃ³n Spark y lectura del dataset.
2. **Etapa 2:** Procesamiento batch vs streaming.
3. **Etapa 3:** Productor y consumidor (simulaciÃ³n de Kafka con una cola de Python).

---
## ETAPA 1 â€” SesiÃ³n Spark y lectura (nivel bÃ¡sico)

In [1]:
import os
import sys
import time
import json
import queue
import shutil
import threading

# Aseguramos que Spark use el mismo intÃ©rprete de Python que este notebook
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Solo en Windows: Spark Streaming necesita winutils.exe (carpeta hadoop/bin del proyecto)
if os.name == "nt":
    os.environ["HADOOP_HOME"] = os.path.abspath("hadoop")
    os.environ["PATH"] = os.path.join(os.environ["HADOOP_HOME"], "bin") + os.pathsep + os.environ["PATH"]

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .master("local[*]")                              # modo local, usando todos los nÃºcleos
    .appName("TallerStreamingKafka")
    .config("spark.sql.shuffle.partitions", "8")     # pocas particiones: dataset mediano, una sola mÃ¡quina
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("VersiÃ³n de Spark:", spark.version)
print("Master:", spark.sparkContext.master)

C:\Users\oscar\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


VersiÃ³n de Spark: 4.2.0
Master: local[*]


**AnÃ¡lisis:** Se importan las librerÃ­as que usaremos en todo el notebook (`time`, `json`, `queue`, `threading` serÃ¡n Ãºtiles en las etapas 2 y 3). Se configuran las variables `PYSPARK_PYTHON` para que los *workers* de Spark usen el mismo Python del notebook (evita errores de versiÃ³n en Windows). AdemÃ¡s, si estamos en Windows, se apunta `HADOOP_HOME` a la carpeta `hadoop/` del proyecto, que contiene `winutils.exe` y `hadoop.dll`: Spark Streaming los necesita en Windows para escribir el *checkpoint* en disco (en Linux/Colab esta parte se omite). Luego se crea la `SparkSession`, que es el punto de entrada a Spark: `master("local[*]")` indica que Spark corre en esta mÃ¡quina usando todos los nÃºcleos disponibles como si fueran nodos del clÃºster. Se reduce `spark.sql.shuffle.partitions` de 200 (valor por defecto) a 8 porque, para un dataset de ~1 millÃ³n de filas en un solo equipo, 200 particiones solo agregarÃ­an sobrecarga. Por Ãºltimo, se baja el nivel de log a `WARN` para no llenar la salida de mensajes informativos.

In [2]:
df = (
    spark.read
    .option("header", True)        # la primera fila contiene los nombres de columna
    .option("inferSchema", True)   # Spark deduce el tipo de dato de cada columna
    .csv("online_retail_II.csv")
)

df.printSchema()

root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- Price: double (nullable = true)
 |-- Customer ID: double (nullable = true)
 |-- Country: string (nullable = true)



**AnÃ¡lisis:** `spark.read.csv` carga el archivo como un **DataFrame** distribuido. Con `header=True` Spark toma la primera lÃ­nea como nombres de columna, y con `inferSchema=True` recorre los datos para deducir los tipos (por eso la lectura tarda un poco mÃ¡s que sin esta opciÃ³n). `printSchema()` muestra el resultado: `Quantity` se detecta como entero, `Price` y `Customer ID` como `double` e `InvoiceDate` como `timestamp`. `Invoice` y `StockCode` quedan como `string` porque tienen valores alfanumÃ©ricos (por ejemplo, facturas `C489449` de cancelaciones o cÃ³digos como `79323P`). Todas las columnas son `nullable = true`, es decir, pueden tener valores nulos (por ejemplo, ventas sin `Customer ID`).

In [3]:
total_filas = df.count()
print(f"NÃºmero de filas del dataset: {total_filas:,}")
print(f"NÃºmero de columnas: {len(df.columns)}")

NÃºmero de filas del dataset: 1,067,371
NÃºmero de columnas: 8


**AnÃ¡lisis:** `count()` es una **acciÃ³n**: obliga a Spark a ejecutar realmente la lectura y contar todas las filas (hasta este punto, las transformaciones eran *lazy*, es decir, solo se planificaban). El dataset tiene algo mÃ¡s de **1 millÃ³n de transacciones** y 8 columnas, un volumen adecuado para ver las ventajas de Spark frente a procesar el archivo lÃ­nea por lÃ­nea.

In [4]:
df.show(5, truncate=False)

+-------+---------+-----------------------------------+--------+-------------------+-----+-----------+--------------+
|Invoice|StockCode|Description                        |Quantity|InvoiceDate        |Price|Customer ID|Country       |
+-------+---------+-----------------------------------+--------+-------------------+-----+-----------+--------------+
|489434 |85048    |15CM CHRISTMAS GLASS BALL 20 LIGHTS|12      |2009-12-01 07:45:00|6.95 |13085.0    |United Kingdom|
|489434 |79323P   |PINK CHERRY LIGHTS                 |12      |2009-12-01 07:45:00|6.75 |13085.0    |United Kingdom|
|489434 |79323W   | WHITE CHERRY LIGHTS               |12      |2009-12-01 07:45:00|6.75 |13085.0    |United Kingdom|
|489434 |22041    |"RECORD FRAME 7"" SINGLE SIZE "    |48      |2009-12-01 07:45:00|2.1  |13085.0    |United Kingdom|
|489434 |21232    |STRAWBERRY CERAMIC TRINKET BOX     |24      |2009-12-01 07:45:00|1.25 |13085.0    |United Kingdom|
+-------+---------+-----------------------------------+-

**AnÃ¡lisis:** `show(5)` trae al *driver* solo las primeras 5 filas y las imprime en formato tabla; `truncate=False` evita que se corten los textos largos (como `Description`). Se observa que cada fila es una lÃ­nea de una factura: la factura `489434` contiene varios productos comprados por el cliente `13085` en el Reino Unido, cada uno con su cantidad y su precio unitario.

In [5]:
df = df.withColumn("Total", F.round(F.col("Quantity") * F.col("Price"), 2))

df.select("Invoice", "Description", "Quantity", "Price", "Total", "Country").show(5, truncate=False)

# InformaciÃ³n extra: devoluciones/cancelaciones (cantidades negativas)
print("Filas con Quantity negativa (devoluciones):", f"{df.filter(F.col('Quantity') < 0).count():,}")

+-------+-----------------------------------+--------+-----+-----+--------------+
|Invoice|Description                        |Quantity|Price|Total|Country       |
+-------+-----------------------------------+--------+-----+-----+--------------+
|489434 |15CM CHRISTMAS GLASS BALL 20 LIGHTS|12      |6.95 |83.4 |United Kingdom|
|489434 |PINK CHERRY LIGHTS                 |12      |6.75 |81.0 |United Kingdom|
|489434 | WHITE CHERRY LIGHTS               |12      |6.75 |81.0 |United Kingdom|
|489434 |"RECORD FRAME 7"" SINGLE SIZE "    |48      |2.1  |100.8|United Kingdom|
|489434 |STRAWBERRY CERAMIC TRINKET BOX     |24      |1.25 |30.0 |United Kingdom|
+-------+-----------------------------------+--------+-----+-----+--------------+
only showing top 5 rows


Filas con Quantity negativa (devoluciones): 22,950


**AnÃ¡lisis:** Como el dataset tiene cantidad (`Quantity`) y precio (`Price`), se crea la columna **`Total = Quantity Ã— Price`** con `withColumn`, redondeada a 2 decimales con `F.round`. `withColumn` no modifica el DataFrame original (los DataFrames son inmutables), sino que devuelve uno nuevo, por eso se reasigna a `df`. Luego se muestran las columnas mÃ¡s relevantes para comprobar el cÃ¡lculo (ej.: 12 Ã— 6.95 = 83.4). AdemÃ¡s se cuentan las filas con cantidad negativa: corresponden a **devoluciones/cancelaciones**, por lo que su `Total` tambiÃ©n es negativo y, al sumar por paÃ­s, restan de las ventas (lo cual es correcto para obtener ventas netas).

---
## ETAPA 2 â€” Batch vs Streaming

### 2.1 Procesamiento en modo **batch**

In [6]:
inicio = time.time()

ventas_por_pais = (
    df.groupBy("Country")
      .agg(
          F.sum("Total").cast("decimal(18,2)").alias("Ventas_Totales"),
          F.countDistinct("Invoice").alias("Num_Facturas"),
          F.count("*").alias("Num_Lineas"),
      )
      .orderBy(F.desc("Ventas_Totales"))
)

ventas_por_pais.show(10, truncate=False)

print(f"PaÃ­ses distintos: {ventas_por_pais.count()}")
print(f"Tiempo de procesamiento batch: {time.time() - inicio:.2f} s")

+--------------+--------------+------------+----------+
|Country       |Ventas_Totales|Num_Facturas|Num_Lineas|
+--------------+--------------+------------+----------+
|United Kingdom|16382583.88   |49108       |981330    |
|EIRE          |615519.55     |806         |17866     |
|Netherlands   |548524.95     |250         |5140      |
|Germany       |417988.56     |1095        |17624     |
|France        |328191.80     |746         |14330     |
|Australia     |167129.07     |117         |1913      |
|Switzerland   |99728.76      |123         |3189      |
|Spain         |91859.48      |188         |3811      |
|Sweden        |87809.42      |129         |1364      |
|Denmark       |65741.09      |53          |817       |
+--------------+--------------+------------+----------+
only showing top 10 rows


PaÃ­ses distintos: 43
Tiempo de procesamiento batch: 2.60 s


**AnÃ¡lisis:** Este es el enfoque **batch**: el dataset completo ya existe en disco, se procesa **una sola vez** y se obtiene un resultado final. `groupBy("Country")` agrupa todas las transacciones por paÃ­s y `agg` calcula, para cada uno, la suma de `Total` (ventas netas, convertida a `decimal(18,2)` para evitar la notaciÃ³n cientÃ­fica), el nÃºmero de facturas distintas (`countDistinct`) y el nÃºmero de lÃ­neas. `orderBy(desc)` ordena de mayor a menor venta. El resultado muestra que **United Kingdom** concentra la gran mayorÃ­a de las ventas (es una tienda britÃ¡nica), seguido muy de lejos por paÃ­ses como EIRE (Irlanda), Netherlands y Germany. Se mide el tiempo total para tener una referencia: Spark procesa mÃ¡s de un millÃ³n de filas en pocos segundos.

### 2.2 Procesamiento en modo **streaming** con la fuente `rate`

In [7]:
stream_df = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 10)   # genera 10 eventos por segundo
    .load()
)

print("Â¿Es un DataFrame de streaming?:", stream_df.isStreaming)
stream_df.printSchema()

Â¿Es un DataFrame de streaming?: True
root
 |-- timestamp: timestamp (nullable = true)
 |-- value: long (nullable = true)



**AnÃ¡lisis:** `spark.readStream` (en lugar de `spark.read`) crea un **DataFrame de streaming**, es decir, una tabla que **crece indefinidamente** a medida que llegan datos. La fuente `rate` es un generador de pruebas incluido en Spark: produce `rowsPerSecond` filas por segundo (aquÃ­ 10), cada una con dos columnas: `timestamp` (momento en que se generÃ³ el evento) y `value` (un contador incremental 0, 1, 2, ...). `isStreaming = True` confirma que no es un DataFrame normal: no se le puede hacer `show()` o `count()` directamente, sino que hay que iniciar una consulta con `writeStream`.

In [8]:
# Borramos checkpoints de ejecuciones anteriores para empezar desde cero
shutil.rmtree("checkpoints", ignore_errors=True)

conteo_eventos = stream_df.groupBy().count()   # cuenta TODOS los eventos recibidos hasta el momento

query = (
    conteo_eventos.writeStream
    .outputMode("complete")                         # en cada micro-batch se reescribe el resultado completo
    .format("memory")                               # el resultado se guarda en una tabla en memoria
    .queryName("conteo_eventos")                    # nombre de esa tabla para consultarla con SQL
    .option("checkpointLocation", "checkpoints/rate_conteo")
    .trigger(processingTime="2 seconds")            # un micro-batch cada 2 segundos
    .start()
)

for i in range(5):
    time.sleep(3)
    resultado = spark.sql("SELECT * FROM conteo_eventos").collect()
    eventos = resultado[0]["count"] if resultado else 0
    print(f"Lectura {i + 1}: {eventos} eventos recibidos hasta ahora")

query.stop()
print("Consulta de streaming detenida. Â¿Activa?:", query.isActive)

Lectura 1: 0 eventos recibidos hasta ahora


Lectura 2: 40 eventos recibidos hasta ahora


Lectura 3: 80 eventos recibidos hasta ahora


Lectura 4: 100 eventos recibidos hasta ahora


Lectura 5: 140 eventos recibidos hasta ahora
Consulta de streaming detenida. Â¿Activa?: False


**AnÃ¡lisis:** AquÃ­ se cuentan los eventos que van llegando. `groupBy().count()` (sin columnas de agrupaciÃ³n) produce un Ãºnico contador global. La consulta se inicia con `writeStream ... start()` y se ejecuta **en segundo plano** de forma continua, en *micro-batches* cada 2 segundos (`trigger`):
- `outputMode("complete")`: en cada micro-batch se emite el resultado acumulado completo (necesario para agregaciones sin ventana).
- `format("memory")` + `queryName`: el resultado se guarda en una tabla en memoria que podemos consultar con `spark.sql`.
- `shutil.rmtree("checkpoints")`: al inicio se borran checkpoints anteriores; si no, Spark retomarÃ­a el estado de una ejecuciÃ³n previa y el conteo no empezarÃ­a en cero.
- `checkpointLocation`: carpeta donde Spark guarda el estado y el progreso (quÃ© datos ya procesÃ³) para poder recuperarse ante fallos.

El bucle consulta la tabla cada 3 segundos y se observa cÃ³mo el **conteo crece** con el tiempo (â‰ˆ10 eventos por segundo): el resultado nunca es "final", siempre se va actualizando. Finalmente `query.stop()` detiene el stream; si no se detuviera, seguirÃ­a corriendo indefinidamente.

In [9]:
# Extra: conteo de eventos por ventanas de tiempo de 5 segundos
conteo_ventanas = (
    stream_df
    .groupBy(F.window("timestamp", "5 seconds"))
    .count()
)

query_ventanas = (
    conteo_ventanas.writeStream
    .outputMode("complete")
    .format("memory")
    .queryName("conteo_ventanas")
    .option("checkpointLocation", "checkpoints/rate_ventanas")
    .start()
)

time.sleep(12)
spark.sql(
    "SELECT window.start AS inicio, window.end AS fin, count FROM conteo_ventanas ORDER BY inicio"
).show(truncate=False)

query_ventanas.stop()

+-------------------+-------------------+-----+
|inicio             |fin                |count|
+-------------------+-------------------+-----+
|2026-10-04 17:12:20|2026-10-04 17:12:25|16   |
|2026-10-04 17:12:25|2026-10-04 17:12:30|50   |
|2026-10-04 17:12:30|2026-10-04 17:12:35|34   |
+-------------------+-------------------+-----+



**AnÃ¡lisis:** Esta celda es un complemento: en lugar de un Ãºnico contador global, se agrupan los eventos en **ventanas de tiempo** de 5 segundos usando `F.window` sobre la columna `timestamp`. Es la forma tÃ­pica de analizar streams ("Â¿cuÃ¡ntas ventas hubo cada 5 minutos?"). Tras ~12 segundos se consulta la tabla y se ve que cada ventana completa tiene â‰ˆ50 eventos (10 por segundo Ã— 5 s); la primera y la Ãºltima pueden tener menos porque la consulta empezÃ³ o terminÃ³ a mitad de la ventana.

### 2.3 ComparaciÃ³n Batch vs Streaming

| Aspecto | Batch (`spark.read`) | Streaming (`spark.readStream`) |
|---|---|---|
| **Datos** | Acotados: el archivo completo ya existe | Ilimitados: llegan continuamente |
| **EjecuciÃ³n** | Una sola vez, termina | Continua (micro-batches), hasta que se detiene |
| **Resultado** | Final y definitivo | Incremental, se actualiza en cada micro-batch |
| **Latencia** | Alta: hay que esperar a tener todos los datos | Baja: los resultados se ven casi en tiempo real |
| **Acciones** | `show()`, `count()`, `collect()` directamente | Requiere `writeStream ... start()` y un *sink* |
| **Estado** | No necesita guardar estado | Guarda estado/progreso en el *checkpoint* |
| **Ejemplo en el taller** | Ventas totales por paÃ­s del histÃ³rico | Conteo de eventos que van llegando |
| **Casos de uso** | Reportes, anÃ¡lisis histÃ³rico, entrenamiento de modelos | Monitoreo, alertas, detecciÃ³n de fraude, dashboards en vivo |

**ConclusiÃ³n:** la API es prÃ¡cticamente la misma (`groupBy`, `agg`, `count`...), lo que cambia es la naturaleza de los datos. En batch respondemos "Â¿cuÃ¡nto se vendiÃ³ en total por paÃ­s?" sobre datos histÃ³ricos; en streaming respondemos "Â¿cuÃ¡ntos eventos estÃ¡n llegando ahora?" y la respuesta cambia constantemente. Spark Structured Streaming trata el stream como una **tabla que crece sin fin**, y por eso permite reutilizar la misma lÃ³gica en ambos modos.

---
## ETAPA 3 â€” Productor y consumidor (Kafka simulado)

### Conceptos (con nuestras palabras)

- **Topic:** es como un "canal" o "buzÃ³n" con nombre donde se publican los mensajes de un mismo tipo (por ejemplo, el topic `ventas`). Los mensajes se guardan en orden de llegada y quedan disponibles para quien quiera leerlos. En Kafka real, un topic se divide en particiones y los mensajes se conservan un tiempo configurado, aunque ya se hayan leÃ­do.
- **Productor:** es la aplicaciÃ³n que **genera y envÃ­a** mensajes al topic. No sabe ni le importa quiÃ©n los va a leer; solo publica. En nuestro caso, el productor toma filas del dataset (ventas) y las publica una por una, como si fueran ventas ocurriendo en tiempo real.
- **Consumidor:** es la aplicaciÃ³n que **se suscribe** al topic y **lee** los mensajes para procesarlos (sumarlos, guardarlos, generar alertas...). Lleva la cuenta de por dÃ³nde va leyendo (en Kafka esto se llama *offset*).
- **Broker:** es el servidor de Kafka que almacena los topics. Gracias a Ã©l, productor y consumidor estÃ¡n **desacoplados**: no se comunican directamente, ni necesitan funcionar al mismo ritmo.

En esta simulaciÃ³n, el **topic** serÃ¡ una `queue.Queue` de Python (estructura FIFO segura para hilos), el **productor** y el **consumidor** serÃ¡n funciones que se ejecutan en hilos (`threading`) diferentes, de forma concurrente.

In [10]:
N_EVENTOS = 50

eventos = [
    {
        "invoice": r["Invoice"],
        "producto": r["Description"],
        "cantidad": r["Quantity"],
        "precio": r["Price"],
        "total": r["Total"],
        "pais": r["Country"],
        "fecha": str(r["InvoiceDate"]),
    }
    for r in (
        df.filter(F.col("Quantity") > 0)
          .select("Invoice", "Description", "Quantity", "Price", "Total", "Country", "InvoiceDate")
          .sample(fraction=0.001, seed=42)   # muestra aleatoria (reproducible) de todo el dataset
          .limit(N_EVENTOS)
          .collect()
    )
]

print(f"Eventos preparados: {len(eventos)}")
print("Ejemplo de evento:")
print(json.dumps(eventos[0], indent=2, ensure_ascii=False))

Eventos preparados: 50
Ejemplo de evento:
{
  "invoice": "490047",
  "producto": "SET OF 12 ROSE BOTANICAL T-LIGHTS",
  "cantidad": 6,
  "precio": 2.95,
  "total": 17.7,
  "pais": "United Kingdom",
  "fecha": "2009-12-03 13:02:00"
}


**AnÃ¡lisis:** Se preparan los eventos que publicarÃ¡ el productor. Del DataFrame de Spark se filtran solo ventas reales (`Quantity > 0`), se seleccionan las columnas Ãºtiles, se toma una muestra aleatoria con `sample` (con `seed` fija para que sea reproducible, y asÃ­ tener ventas de distintos paÃ­ses y fechas) y se limitan a 50 filas con `limit` y se traen al driver con `collect()` (cuidado: `collect()` solo debe usarse con pocos datos, porque trae todo a la memoria local). Cada fila se convierte en un **diccionario de Python**, que es el formato tÃ­pico de un mensaje (en Kafka normalmente se envÃ­a como JSON). La fecha se pasa a texto para que sea serializable en JSON. Se imprime un ejemplo para ver la estructura del mensaje.

In [11]:
# El "topic" de Kafka simulado con una cola FIFO segura para hilos
topic_ventas = queue.Queue()

FIN_STREAM = None  # marcador especial para avisar al consumidor que ya no hay mÃ¡s mensajes


def productor(topic, eventos, pausa=0.02):
    """Publica cada evento en el topic como un mensaje JSON."""
    for offset, evento in enumerate(eventos):
        mensaje = json.dumps(evento)       # serializamos, como harÃ­a un productor de Kafka
        topic.put(mensaje)                 # publicar en el topic
        if offset < 3:
            print(f"[PRODUCTOR] offset={offset} publicado: {evento['invoice']} | {evento['producto']} | {evento['total']}")
        time.sleep(pausa)                  # simulamos que las ventas llegan poco a poco
    print(f"[PRODUCTOR] Terminado: {len(eventos)} mensajes publicados")
    topic.put(FIN_STREAM)

**AnÃ¡lisis:** Se crea el **topic simulado** con `queue.Queue()`: una cola FIFO (el primer mensaje en entrar es el primero en salir) y *thread-safe*, es decir, varios hilos pueden escribir y leer de ella sin corromper los datos. La funciÃ³n `productor` recorre los eventos y, para cada uno, lo **serializa a JSON** (los mensajes de Kafka viajan como bytes/texto) y lo **publica** con `put`. El Ã­ndice `offset` imita el nÃºmero de posiciÃ³n que Kafka asigna a cada mensaje dentro de una particiÃ³n. La pequeÃ±a pausa `sleep` simula que las ventas ocurren a lo largo del tiempo y no todas de golpe. Al final se publica un marcador `None` para indicar el fin del stream (en Kafka real el stream no termina; esto solo sirve para cerrar la simulaciÃ³n).

In [12]:
resultados = {}


def consumidor(topic, resultados):
    """Lee los mensajes del topic y acumula las ventas."""
    total_ventas = 0.0
    mensajes_leidos = 0
    ventas_por_pais = {}

    while True:
        mensaje = topic.get()              # bloquea hasta que llegue un mensaje
        if mensaje is FIN_STREAM:
            break
        evento = json.loads(mensaje)       # deserializamos el JSON
        total_ventas += evento["total"]
        mensajes_leidos += 1
        ventas_por_pais[evento["pais"]] = ventas_por_pais.get(evento["pais"], 0) + evento["total"]

        if mensajes_leidos % 10 == 0:
            print(f"[CONSUMIDOR] {mensajes_leidos} mensajes leÃ­dos | acumulado = {total_ventas:,.2f}")

    resultados["total_ventas"] = round(total_ventas, 2)
    resultados["mensajes_leidos"] = mensajes_leidos
    resultados["ventas_por_pais"] = {p: round(v, 2) for p, v in ventas_por_pais.items()}
    print("[CONSUMIDOR] Fin del stream recibido")

**AnÃ¡lisis:** La funciÃ³n `consumidor` representa la aplicaciÃ³n suscrita al topic. En un bucle infinito hace `topic.get()`, que **espera (bloquea)** hasta que haya un mensaje disponible, igual que un consumidor de Kafka que hace *poll* esperando datos nuevos. Cada mensaje se **deserializa** con `json.loads` y se procesa: se suma el `total` al acumulado general, se cuenta el mensaje y se acumula el total por paÃ­s en un diccionario. Cada 10 mensajes imprime el progreso, mostrando que el resultado se va **actualizando incrementalmente** (como en streaming). Cuando recibe el marcador de fin, sale del bucle y guarda los resultados en el diccionario `resultados` para poder consultarlos desde fuera del hilo.

In [13]:
hilo_productor = threading.Thread(target=productor, args=(topic_ventas, eventos))
hilo_consumidor = threading.Thread(target=consumidor, args=(topic_ventas, resultados))

inicio = time.time()
hilo_consumidor.start()   # el consumidor arranca primero y queda esperando mensajes
hilo_productor.start()

hilo_productor.join()
hilo_consumidor.join()

print("\n===== RESULTADO DEL CONSUMIDOR =====")
print(f"Mensajes leÃ­dos : {resultados['mensajes_leidos']}")
print(f"Total de ventas : {resultados['total_ventas']:,.2f}")
print("Ventas por paÃ­s :", resultados["ventas_por_pais"])
print(f"Tiempo          : {time.time() - inicio:.2f} s")

[PRODUCTOR] offset=0 publicado: 490047 | SET OF 12 ROSE BOTANICAL T-LIGHTS | 17.7
[PRODUCTOR] offset=1 publicado: 490053 | RED SPOT HEART HOT WATER BOTTLE | 4.95
[PRODUCTOR] offset=2 publicado: 490071 | CREAM SWEETHEART TRAYS | 12.75
[CONSUMIDOR] 10 mensajes leÃ­dos | acumulado = 93.43


[CONSUMIDOR] 20 mensajes leÃ­dos | acumulado = 225.47
[CONSUMIDOR] 30 mensajes leÃ­dos | acumulado = 385.83


[CONSUMIDOR] 40 mensajes leÃ­dos | acumulado = 532.82


[CONSUMIDOR] 50 mensajes leÃ­dos | acumulado = 613.37
[PRODUCTOR] Terminado: 50 mensajes publicados
[CONSUMIDOR] Fin del stream recibido

===== RESULTADO DEL CONSUMIDOR =====
Mensajes leÃ­dos : 50
Total de ventas : 613.37
Ventas por paÃ­s : {'United Kingdom': 553.37, 'Portugal': 16.5, 'France': 29.7, 'EIRE': 13.8}
Tiempo          : 1.03 s


**AnÃ¡lisis:** AquÃ­ productor y consumidor se ejecutan **al mismo tiempo**, cada uno en su propio hilo (`threading.Thread`). El consumidor se inicia primero y queda bloqueado esperando; en cuanto el productor publica, el consumidor lee y procesa. En la salida se ven intercalados los mensajes `[PRODUCTOR]` y `[CONSUMIDOR]`, lo que demuestra que trabajan de forma **concurrente y desacoplada**: ninguno llama al otro directamente, solo se comunican a travÃ©s del topic (la cola). `join()` espera a que ambos hilos terminen antes de imprimir el resultado final: nÃºmero de mensajes leÃ­dos, la suma total de las ventas y el desglose por paÃ­s.

In [14]:
# VerificaciÃ³n: la suma del consumidor debe coincidir con la suma calculada directamente
suma_esperada = round(sum(e["total"] for e in eventos), 2)

print(f"Suma calculada por el consumidor : {resultados['total_ventas']:,.2f}")
print(f"Suma directa de los eventos      : {suma_esperada:,.2f}")
print("Â¿Coinciden?:", abs(resultados["total_ventas"] - suma_esperada) < 0.01)
print("Â¿Se leyeron todos los mensajes?:", resultados["mensajes_leidos"] == len(eventos))
print("Â¿Topic vacÃ­o al final?:", topic_ventas.empty())

Suma calculada por el consumidor : 613.37
Suma directa de los eventos      : 613.37
Â¿Coinciden?: True
Â¿Se leyeron todos los mensajes?: True
Â¿Topic vacÃ­o al final?: True


**AnÃ¡lisis:** Se comprueba que la simulaciÃ³n funcionÃ³ correctamente: la suma acumulada por el consumidor coincide con la suma directa de los 50 eventos producidos, se leyeron todos los mensajes (no se perdiÃ³ ninguno) y el topic quedÃ³ vacÃ­o. Una diferencia con Kafka real: en `queue.Queue` el mensaje **desaparece** al leerlo con `get()`, mientras que en Kafka los mensajes **permanecen** en el topic y cada consumidor (o grupo de consumidores) mantiene su propio *offset*, por lo que varios consumidores pueden leer los mismos datos de forma independiente o incluso re-leerlos.

---
## Conclusiones

- **Etapa 1:** La `SparkSession` en modo local permite leer un CSV de mÃ¡s de un millÃ³n de filas, inferir su esquema y crear columnas derivadas (`Total`) con pocas lÃ­neas de cÃ³digo.
- **Etapa 2:** El modo **batch** procesa datos histÃ³ricos completos y da un resultado final (ventas por paÃ­s, con United Kingdom a la cabeza). El modo **streaming** procesa datos que llegan continuamente y da resultados que se actualizan en cada micro-batch. Spark usa la misma API para ambos.
- **Etapa 3:** Con una cola de Python simulamos el patrÃ³n de Kafka: un **productor** publica eventos (ventas) en un **topic** y un **consumidor** los lee y los suma en tiempo real, ambos desacoplados y trabajando en paralelo. Esta es la base de las arquitecturas de streaming reales, donde Kafka actÃºa como intermediario y Spark Structured Streaming como consumidor/procesador.

In [15]:
spark.stop()
print("SparkSession detenida")

SparkSession detenida


**AnÃ¡lisis:** Se detiene la `SparkSession` para liberar los recursos (memoria, hilos y la JVM de Spark). Es una buena prÃ¡ctica cerrarla al terminar el trabajo.